In [3]:
# imports
from gqlalchemy import Memgraph
import pandas as pd

In [4]:
# connection to graph
memgraph = Memgraph("127.0.0.1", 7687)

In [5]:
# creation of functions for easy Cypher running via VSC

# run and create df
def run_df(query):
    return pd.DataFrame(memgraph.execute_and_fetch(query))


# run and just print
def run_print(query, limit=None):
    results = list(memgraph.execute_and_fetch(query))

    if limit:
        results = results[:limit]

    for row in results:
        print(row)

In [6]:
# Lowie's aviation and legal actions used to construct avation and legal networks
AVIATION_ACTIONS = [
    'traveled with', 'accompanied', 'flew', 'flew with', 'flew on',
    'flew to', 'traveled on aircraft with', 'traveled on private jet of',
    'flew as passenger on', 'traveled on'
]

LEGAL_ACTIONS = [
    'testified before', 'investigated', 'accused', 'arrested', 'charged',
    'prosecuted', 'negotiated non-prosecution agreement with',
    'signed non-prosecution agreement with', 'was charged with', 'was arrested by'
]

## 1.Step 1: "Ground truth" legal actions 
### 1.a. problem in Lowie's analysis: Ghislaine Maxwell (GM) missing
GM is missing in Lowie's bridge notes between his aviation and legal network. 
    - She however appears in aviation network with 15 connection 
    - it is public knowledge that she has been convicted for 20 years. 


#### 1.a.1. as lowie's code for legal network limits to the top 20 
=> first check if GM has any legal actions defined in Lowie's legal action list (she might fall out of top 20) 

In [7]:
run_df("""
MATCH (p:Entity)-[r:RELATED_TO]-(other:Entity)
WHERE p.name = 'Ghislaine Maxwell'
  AND r.action IN ['testified before', 'investigated', 'accused', 'arrested', 'charged',
                   'prosecuted', 'negotiated non-prosecution agreement with',
                   'signed non-prosecution agreement with', 'was charged with', 'was arrested by']
RETURN p.name AS entity, count(*) AS legal_connections
""")

""


RESULT: 0 => GM has no legal actions as defined in Lowie's legal action list

#### 1.a.2. Which RELATED_TO actions are linked to GM in the graph? (top 200, by frequency)
(searching undirected relationships, providing GM both as target and actor of action. specific role is specified in return for each action)

In [14]:
run_df("""
MATCH (p:Entity)-[r:RELATED_TO]-(other:Entity)
WHERE p.name = 'Ghislaine Maxwell'
RETURN DISTINCT r.action AS action, 
    CASE WHEN startNode(r) = p THEN 'actor' ELSE 'target' END AS role,
    count(*) AS n
ORDER BY n DESC
LIMIT 100
""")

,action,role,n
0,sent email to,target,34
1,recruited,actor,17
2,sent email to,actor,14
3,introduced,actor,14
4,attended,actor,11
...,...,...,...
95,testified about,actor,2
96,inquired about service on,target,2
97,directed victim to have sex with,actor,2
98,interviewed for employment,actor,2


RESULT: list fed to Claude, to make a selection of actions with legal nature => 3 categories returned:

(1) GM as defendant (n=count):  
"served subpoena to" (6) 
"served subpoena on" (3) 
"served with subpoena" (2) 
"challenged exclusion of evidence regarding plaintiff's" (6) 
"opposed motions in limine filed by" (2) 
"argued before court regarding" (2) 
"rescheduled deposition of" (3) 
"scheduled deposition for" (2) 
"attempted to depose" (2) 
"sought to introduce evidence regarding" (2) 
"invoked privilege against self-incrimination" (1) 
"accused of participating in" (2)

(2) GM as victim/plaintiff (n=count):
"sued" (7) 
"filed defamation lawsuit against" (5)
"filed defamation suit against" (4) 
"filed civil lawsuit against" (2) 
"filed federal civil suit against" (2)
"filed lawsuit against" (2)
"filed suit against" (2) 
"filed complaint against" (2) 
"sued for defamation" (2) 
"filed legal brief as attorney for" (2) 

(3) other (can be both as defendant or victim/plaintiff): 
"questioned" (3)
"interviewed" (3) 
"testified about" (2)
"identified" (3) 
"settled lawsuit with" (5)

CONCLUSIONS: 
    - GM's criminal conviction seems to be virtually absent from the data. Possible explanation: GM was convicted in DEC 2021. However many of the released documents by the house of representation cover the periode before her arrest in 2020. In that period GM had civil law suites with e.g. Giuffre: Giuffre v. Maxwell (2015-2017) (Giuffre filed a civil defamation lawsuit against GM, alleging GM had called her claims about Epstein's trafficking 'obvious lies'. The case was settled in 2017.)

    - vs. Lowie's legal_actions list: 
            (1) civil vs criminal law: LLM seems to have used terminology of civil law suites rather than those of criminal law suites, for GM. Lowie's legal network focussed mainly on criminal law terminology <=> we need to rebuild the legal actions list. We will examine RELATED_TO actions linked to other figures with documented legal involvement in the Epstein case (public knowledge) (our ground truth figures) => we use these to extend legal actions list with both civil and criminal terms as they actually appear in the dataset.

            (2) searching with IN vs CONTAINS in WHERE statement: e.g. "accused" part of Lowie's list, however due to the use of IN in the where statement, Cypher only searches for exact matches. => search made anew, using CONTAIN instead of IN in WHERE statement. (including other legal_actions from Lowie's list)

In [18]:
run_print("""
MATCH (p:Entity)-[r:RELATED_TO]-(other:Entity)
WHERE p.name = 'Ghislaine Maxwell'
AND (
    toLower(r.action) CONTAINS 'testified'
    OR toLower(r.action) CONTAINS 'investigated'
    OR toLower(r.action) CONTAINS 'accused'
    OR toLower(r.action) CONTAINS 'arrested'
    OR toLower(r.action) CONTAINS 'charged'
    OR toLower(r.action) CONTAINS 'prosecuted'
    OR toLower(r.action) CONTAINS 'non-prosecution'
    OR toLower(r.action) CONTAINS 'charged'
)
RETURN r.action AS action, count(*) AS n
ORDER BY n DESC
""")

{'action': 'accused of participating in', 'n': 2}
{'action': 'testified about', 'n': 2}
{'action': 'accused of procuring', 'n': 1}
{'action': 'testified to', 'n': 1}
{'action': 'testified that she worked for', 'n': 1}
{'action': 'accused of helping recruit and sexually exploit', 'n': 1}
{'action': 'testified at deposition regarding knowledge about', 'n': 1}
{'action': 'testified', 'n': 1}
{'action': 'testified that she did not look at', 'n': 1}
{'action': 'testified that she was unsure about', 'n': 1}
{'action': 'testified at deposition that she never read prior to January 2015', 'n': 1}
{'action': 'testified regarding relationship with', 'n': 1}


RESULTS: GM would have appeared in Lowie's legal network if search would have been with CONTAIN in WHERE statement. 

CONCLUSION: 
(2) important to search with CONTAIN in WHERE statement in legal_network, and to adapt terminology accordingly. 

### 1.b Rebuilding legal action list: analyse RELATED_TO legal actions of known figures with publicly known legal involvement linked to Epstein's operations 
(based on public record, verified against the data):

Chosen public figures: 
Jeffrey Epstein — arrested 2019, died in custody
Ghislaine Maxwell — convicted (20 years, 2021)
Jean-Luc Brunel — arrested, died in custody (2022)
Alfredo Rodriguez — convicted of obstruction (18 months)
Virginia Roberts Giuffre — central plaintiff in multiple civil proceedings
Prince Andrew — settled civil lawsuit with Giuffre (2022)
Alan Dershowitz — named in civil proceedings by Giuffre
Sarah Kellen — named co-conspirator in 2008 NPA, named in civil lawsuits
Adriana Ross — named co-conspirator in 2008 NPA, named in civil lawsuits
Nadia Marcinkova — named co-conspirator in 2008 NPA, named in civil lawsuits
Lesley Groff — named co-conspirator in 2008 NPA, named in civil lawsuits

#### 1.b.1 For each figure: verify name variants, aliases, and possible duplicate nodes in the dataset

For example: for Epstein himself. 

In [19]:
# looking for all entities that might represent epstein himself in the graph
run_print("""
MATCH (e:Entity)
WHERE e.name = 'Jeffrey Epstein'
   OR e.name = 'Epstein'
   OR e.name = 'J. Epstein'
   OR e.name = 'Jeff Epstein'
RETURN e.name, e.entity_type, count(*) AS n
""")

{'e.name': 'Jeffrey Epstein', 'e.entity_type': 'victim', 'n': 1}
{'e.name': 'Jeff Epstein', 'e.entity_type': None, 'n': 1}
{'e.name': 'Epstein', 'e.entity_type': None, 'n': 1}


In [20]:
# looking for all aliases that might be linked to these entity nodes
run_print("""
MATCH (a:Alias)-[:ALIAS_OF]->(e:Entity)
WHERE e.name IN ['Jeffrey Epstein', 'Jeff Epstein', 'Epstein']
RETURN e.name AS canonical_name, a.name AS alias
ORDER BY e.name
""")

{'canonical_name': 'Epstein', 'alias': 'Epstein'}
{'canonical_name': 'Jeff Epstein', 'alias': 'Jeff Epstein'}
{'canonical_name': 'Jeffrey Epstein', 'alias': 'Epstein Jeffrey'}
{'canonical_name': 'Jeffrey Epstein', 'alias': 'J'}
{'canonical_name': 'Jeffrey Epstein', 'alias': 'Jeff'}
{'canonical_name': 'Jeffrey Epstein', 'alias': 'Jeffrey'}
{'canonical_name': 'Jeffrey Epstein', 'alias': 'Jeffrey E.'}
{'canonical_name': 'Jeffrey Epstein', 'alias': 'Jeffrey E. Epstein'}
{'canonical_name': 'Jeffrey Epstein', 'alias': 'Jeffrey Edward Epstein'}
{'canonical_name': 'Jeffrey Epstein', 'alias': 'Jeffrey Epstein'}
{'canonical_name': 'Jeffrey Epstein', 'alias': 'jee'}


RESULTS: 
3 entity nodes that represent epstein. Only to one of them aliases are linked (looking at the different amount of Aliases linked, first node herunder might be kind of the "main Epstein node").
    (1) "Jeffrey Epstein" - with aliases: 
        'Jeffrey Edward Epstein', 'Jeffrey E. Epstein', 'Jeffrey E.', 'Jeffrey', 'Jeff', 'J', 'jee', 'Epstein Jeffrey'
    (2) "Jeff Epstein"   (no aliases)
    (3) "Epstein" (no aliases)

In [24]:
# search for our other "ground truth"-figures: 
run_print("""
MATCH (a:Alias)-[:ALIAS_OF]->(e:Entity)
WHERE toLower(e.name) CONTAINS 'maxwell'
   OR toLower(e.name) CONTAINS 'brunel'
   OR toLower(e.name) CONTAINS 'rodriguez'
   OR toLower(e.name) CONTAINS 'giuffre'
   OR toLower(e.name) CONTAINS 'virginia roberts'
   OR toLower(e.name) CONTAINS 'prince andrew'
   OR toLower(e.name) CONTAINS 'dershowitz'
   OR toLower(e.name) CONTAINS 'kellen'
   OR toLower(e.name) CONTAINS 'marcinkova'
   OR toLower(e.name) CONTAINS 'marcinko'
   OR toLower(e.name) CONTAINS 'lesley groff'
   OR toLower(e.name) CONTAINS 'adriana'
RETURN e.name AS canonical_name, a.name AS alias
ORDER BY e.name
""")

{'canonical_name': 'Adriana Mucinska', 'alias': 'Adriana Mucinska'}
{'canonical_name': 'Adriana Mucinska Ross', 'alias': 'Adriana Mucinska Ross'}
{'canonical_name': 'Adriana Mucinska Ross', 'alias': 'Adriana Ross'}
{'canonical_name': 'Alan Dershowitz and Jeffrey Epstein', 'alias': 'Alan Dershowitz and Jeffrey Epstein'}
{'canonical_name': 'Alan Dershowitz and Matthew Hale', 'alias': 'Alan Dershowitz and Matthew Hale'}
{'canonical_name': 'Alan Dershowitz and Prince Andrew', 'alias': 'Alan Dershowitz and Prince Andrew'}
{'canonical_name': 'Alan Dershowitz and Rebecca Boylan', 'alias': 'Alan Dershowitz and Rebecca Boylan'}
{'canonical_name': 'Alan Dershowitz and Tatiana', 'alias': 'Alan Dershowitz and Tatiana'}
{'canonical_name': 'Alan Dershowitz and brother Nathan', 'alias': 'Alan Dershowitz and brother Nathan'}
{'canonical_name': 'Alan Dershowitz and classmate', 'alias': 'Alan Dershowitz and classmate'}
{'canonical_name': 'Alan Dershowitz and family', 'alias': 'Alan Dershowitz and family

RESULTS: 
- List was fed to Claude to filter out relevant nodes (referring to actual persons): 
    (1) GM: 
    'Ghislaine Maxwell' — aliases: 'Maxwell', 'Ghislaine'
    'G Maxwell' — no alias
    
    (2) Brunel:
    'Jean-Luc Didier Henri-Rene Brunel' — aliases: 'Jean-Luc Brunel', 'Brunel'
    'Jean Luc Brunel' — aliases: 'Jean Luc Bruhnel', 'Jean Luc Brune'
    'Jean-Luc Brunel' — no alias

    (3) Rodriguez:
    'Alfredo Rodriguez' — aliases: 'Alfredo Rodriguez (household account manager)', 'Fred'
    'Alfonso Rodriguez' — no alias (possibly a duplicate)

    (4) Giuffre:
    'Virginia Roberts Giuffre' — aliases: 'Virginia Roberts', 'Virginia L. Giuffre', 'Virginia Roberts (age 17)' (and more)
    'Virginia L. Giuffre' — aliases: 'Virginia Giuffre', 'Virginia Giuffre/Roberts'
    'Virginia' - no aliases

    (5) Prince Andrew:
    'Prince Andrew, Duke of York' — aliases: 'Andrew', 'Prince Andrew', 'Prince Andrew (Duke of York)', (and more)

    (6) Dershowitz:
    'Alan M. Dershowitz' — aliases: 'Alan Dershowitz', 'Dershowitz', 'Alan', 'Professor Alan Dershowitz'
    'Professor Dershowitz' — no aliases

    (7) Kellen:
    'Sarah Kellen' — aliases: 'Sarah', 'Kellen'

    (8) Ross:
    'Adriana Mucinska Ross' — aliases: 'Adriana Ross'
    'Adriana Mucinska' — no aliases
    
    (9) Marcinkova:
    'Nadia Marcinkova' — aliases: 'Nadia', 'Marcinkova'
    'Nada Marcinkova' — no aliases
    'Natalia Marcinkova' — no aliases

    (10) Groff:
    'Lesley Groff' — aliases: 'Lesley'


CONCLUSION: 
- our ground truth figures list with all nodes: 


In [28]:
ground_truth_figures = {
    'Epstein': ['Jeffrey Epstein', 'Jeff Epstein', 'Epstein'],
    'Maxwell': ['Ghislaine Maxwell', 'G Maxwell'],
    'Brunel': ['Jean-Luc Brunel', 'Jean Luc Brunel', 
               'Jean-Luc Didier Henri-Rene Brunel'],
    'Rodriguez': ['Alfredo Rodriguez', 'Alfonso Rodriguez'],
    'Giuffre': ['Virginia Roberts Giuffre', 'Virginia L. Giuffre', 'Virginia'],
    'Prince Andrew': ['Prince Andrew, Duke of York'],
    'Dershowitz': ['Alan M. Dershowitz', 'Professor Dershowitz'],
    'Kellen': ['Sarah Kellen'],
    'Ross': ['Adriana Mucinska Ross', 'Adriana Mucinska'],
    'Marcinkova': ['Nadia Marcinkova', 'Nada Marcinkova', 'Natalia Marcinkova'],
    'Groff': ['Lesley Groff']
}

#### 1.b.2. Extract their legal actions from the graph to build an empirically grounded, extended LEGAL_ACTIONS list

In [31]:
# extracting for each ground_truth_figure top 200 actions (in general) first: 
results = []
for group, names in ground_truth_figures.items():
    for name in names:
        df = run_print(f"""
        MATCH (p:Entity)-[r:RELATED_TO]-(other:Entity)
        WHERE p.name = '{name}'
        RETURN p.name AS person, r.action AS action, count(*) AS n
        ORDER BY n DESC
        LIMIT 300
        """)

{'person': 'Jeffrey Epstein', 'action': 'sent email to', 'n': 5426}
{'person': 'Jeffrey Epstein', 'action': 'sent message to', 'n': 1063}
{'person': 'Jeffrey Epstein', 'action': 'met with', 'n': 243}
{'person': 'Jeffrey Epstein', 'action': 'forwarded email to', 'n': 230}
{'person': 'Jeffrey Epstein', 'action': 'commented on', 'n': 158}
{'person': 'Jeffrey Epstein', 'action': 'sent email response to', 'n': 140}
{'person': 'Jeffrey Epstein', 'action': 'paid', 'n': 120}
{'person': 'Jeffrey Epstein', 'action': 'inquired about', 'n': 116}
{'person': 'Jeffrey Epstein', 'action': 'mentioned', 'n': 110}
{'person': 'Jeffrey Epstein', 'action': 'owns', 'n': 108}
{'person': 'Jeffrey Epstein', 'action': 'pleaded guilty to', 'n': 106}
{'person': 'Jeffrey Epstein', 'action': 'informed', 'n': 98}
{'person': 'Jeffrey Epstein', 'action': 'shared article about', 'n': 95}
{'person': 'Jeffrey Epstein', 'action': 'issued confidentiality notice', 'n': 94}
{'person': 'Jeffrey Epstein', 'action': 'asserted ow

## 2.Step 2: bridge nodes analysis (as Lowie did before)
### 2.a. Who traveled with Epstein specifically? 
Search with edges: undirected, deduplicated.  
Travel network = proxy for social network. 
Travel connections directly linked to Epstein himself. NOTE: use of Aliases and duplicate nodes of Epstein himself. 

## EXPLORATORY elements

In [26]:
# which entity types are actually exiting? 
run_print("""
MATCH (e:Entity)
WHERE e.entity_type IS NOT NULL
RETURN DISTINCT e.entity_type
ORDER BY e.entity_type
LIMIT 100
""")

{'e.entity_type': 'BDSM practitioner'}
{'e.entity_type': 'Chinese publisher'}
{'e.entity_type': 'NSA staff member'}
{'e.entity_type': 'NSA_security_officer'}
{'e.entity_type': 'Palestinian assailant'}
{'e.entity_type': 'Soviet_defector'}
{'e.entity_type': 'abuser'}
{'e.entity_type': 'academic'}
{'e.entity_type': 'academic administrator'}
{'e.entity_type': 'academic or activist'}
{'e.entity_type': 'academic or researcher'}
{'e.entity_type': 'academic researcher'}
{'e.entity_type': 'academic researcher or technology theorist'}
{'e.entity_type': 'academic researchers'}
{'e.entity_type': 'academic_administrator'}
{'e.entity_type': 'academic_author'}
{'e.entity_type': 'academic_economist'}
{'e.entity_type': 'academic_expert'}
{'e.entity_type': 'academic_faculty'}
{'e.entity_type': 'academic_lecturer'}
{'e.entity_type': 'academic_mentor'}
{'e.entity_type': 'academic_or_policy_analyst'}
{'e.entity_type': 'academic_or_scientist'}
{'e.entity_type': 'academic_researcher'}
{'e.entity_type': 'acad